# Solutions - Visualisation de données avec Plotly (Introduction)

In [ ]:
# Importer les bibliothèques nécessaires
import pandas as pd
import numpy as np

# Importer Plotly
import plotly
import plotly.express as px  # Interface simplifiée de Plotly
import plotly.graph_objects as go  # Interface plus complète de Plotly
from plotly.subplots import make_subplots  # Pour créer des sous-graphiques

# Afficher les versions des bibliothèques
print("Pandas version:", pd.__version__)
print("Plotly version:", plotly.__version__)

In [ ]:
# Charger le jeu de données
df = pd.read_csv('../../data/passenger_satisfaction/train_50.csv')

# Afficher les premières lignes
df.head()

## Exercice 1
Créez un histogramme montrant la distribution des retards au départ ('Departure Delay in Minutes'). Utilisez un binning approprié et ajoutez un titre et des labels d'axes.

In [ ]:
# Créer un histogramme pour les retards au départ
fig = px.histogram(
    df, 
    x='Departure Delay in Minutes',
    title='Distribution des retards au départ',
    labels={'Departure Delay in Minutes': 'Retard au départ (minutes)', 'count': 'Nombre de vols'},
    nbins=50,  # Nombre de bins approprié
    color_discrete_sequence=['#FF7F0E']  # Couleur orange
)

# Personnaliser le graphique
fig.update_layout(
    height=500,
    width=800,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    title_font_size=18
)

fig.show()

## Exercice 2
Créez un graphique à barres montrant le nombre de passagers par type de client ('Customer Type') et par genre ('Gender'). Utilisez la couleur pour distinguer les genres.

In [ ]:
# Compter le nombre de passagers par type de client et genre
customer_gender_counts = df.groupby(['Customer Type', 'Gender']).size().reset_index(name='Count')

# Créer un graphique à barres groupées
fig = px.bar(
    customer_gender_counts, 
    x='Customer Type', 
    y='Count', 
    color='Gender',
    title='Nombre de passagers par type de client et genre',
    labels={'Customer Type': 'Type de client', 'Count': 'Nombre de passagers', 'Gender': 'Genre'},
    barmode='group',  # Barres groupées (au lieu d'empilées)
    color_discrete_map={'Male': '#636EFA', 'Female': '#EF553B'}  # Couleurs personnalisées
)

# Personnaliser le graphique
fig.update_layout(
    height=500,
    width=800,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    title_font_size=18,
    legend_title_font_size=14
)

fig.show()

## Exercice 3
Créez un graphique en camembert montrant la répartition des classes de voyage ('Class'). Assurez-vous d'afficher les pourcentages et les labels.

In [ ]:
# Calculer la répartition des classes de voyage
class_counts = df['Class'].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']

# Créer un graphique en camembert
fig = px.pie(
    class_counts, 
    names='Class', 
    values='Count',
    title='Répartition des classes de voyage',
    color='Class',  # Colorer selon la classe
    color_discrete_sequence=px.colors.qualitative.Pastel  # Palette de couleurs
)

# Personnaliser le graphique
fig.update_traces(
    textinfo='percent+label',  # Afficher les pourcentages et les labels
    hoverinfo='label+percent+value',  # Info au survol
    textfont_size=14,  # Taille de la police
    marker=dict(line=dict(color='#FFFFFF', width=2))  # Bordure blanche
)

fig.update_layout(
    height=600,
    width=800,
    title_font_size=18,
    legend_title_font_size=14
)

fig.show()

## Bonus : Tableau de bord complet

Créons un tableau de bord complet qui combine plusieurs visualisations pour analyser la satisfaction des passagers.

In [ ]:
# Créer un tableau de bord avec 4 graphiques
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Satisfaction par classe',
        'Satisfaction par type de client',
        'Impact du retard sur la satisfaction',
        'Satisfaction par genre et âge'
    ),
    specs=[
        [{'type': 'bar'}, {'type': 'bar'}],
        [{'type': 'bar'}, {'type': 'bar'}]
    ]
)

# 1. Satisfaction par classe
class_satisfaction = pd.crosstab(df['Class'], df['Satisfaction'], normalize='index') * 100
for satisfaction in class_satisfaction.columns:
    fig.add_trace(
        go.Bar(
            x=class_satisfaction.index,
            y=class_satisfaction[satisfaction],
            name=satisfaction,
            marker_color='#636EFA' if satisfaction == 'satisfied' else '#EF553B'
        ),
        row=1, col=1
    )

# 2. Satisfaction par type de client
customer_satisfaction = pd.crosstab(df['Customer Type'], df['Satisfaction'], normalize='index') * 100
for satisfaction in customer_satisfaction.columns:
    fig.add_trace(
        go.Bar(
            x=customer_satisfaction.index,
            y=customer_satisfaction[satisfaction],
            name=satisfaction,
            marker_color='#636EFA' if satisfaction == 'satisfied' else '#EF553B',
            showlegend=False  # Ne pas répéter la légende
        ),
        row=1, col=2
    )

# 3. Impact du retard sur la satisfaction
# Créer une variable pour les vols retardés
df['Flight_Delayed'] = df['Departure Delay in Minutes'] > 0
df['Flight_Delayed'] = df['Flight_Delayed'].map({True: 'Vol retardé', False: 'Vol à l\'heure'})
delay_satisfaction = pd.crosstab(df['Flight_Delayed'], df['Satisfaction'], normalize='index') * 100
for satisfaction in delay_satisfaction.columns:
    fig.add_trace(
        go.Bar(
            x=delay_satisfaction.index,
            y=delay_satisfaction[satisfaction],
            name=satisfaction,
            marker_color='#636EFA' if satisfaction == 'satisfied' else '#EF553B',
            showlegend=False  # Ne pas répéter la légende
        ),
        row=2, col=1
    )

# 4. Satisfaction par genre et âge
# Créer des catégories d'âge
df['Age_Category'] = pd.cut(df['Age'], 
                           bins=[0, 18, 35, 50, 65, 100], 
                           labels=['<18', '18-35', '36-50', '51-65', '>65'])
gender_age_satisfaction = df.groupby(['Gender', 'Age_Category'])['Satisfaction'].apply(
    lambda x: (x == 'satisfied').mean() * 100
).reset_index(name='Satisfaction_Rate')

# Filtrer pour chaque genre
for gender in ['Male', 'Female']:
    gender_data = gender_age_satisfaction[gender_age_satisfaction['Gender'] == gender]
    fig.add_trace(
        go.Bar(
            x=gender_data['Age_Category'],
            y=gender_data['Satisfaction_Rate'],
            name=gender,
            marker_color='#636EFA' if gender == 'Male' else '#EF553B'
        ),
        row=2, col=2
    )

# Personnaliser le tableau de bord
fig.update_layout(
    title_text="Analyse complète de la satisfaction des passagers",
    height=800,
    width=1200,
    barmode='group'
)

# Mettre à jour les axes
for i in range(1, 3):
    for j in range(1, 3):
        fig.update_yaxes(title_text='Pourcentage (%)', range=[0, 100], row=i, col=j)

fig.update_xaxes(title_text='Classe', row=1, col=1)
fig.update_xaxes(title_text='Type de client', row=1, col=2)
fig.update_xaxes(title_text='Statut du vol', row=2, col=1)
fig.update_xaxes(title_text='Catégorie d\'âge', row=2, col=2)

fig.show()